In [4]:
import os
os.chdir("/home/ec2-user/SageMaker/")
print(f"Working directory saat ini: {os.getcwd()}")

Working directory saat ini: /home/ec2-user/SageMaker


In [5]:
import boto3
import sagemaker

sm_session = sagemaker.Session()
BUCKET = sm_session.default_bucket() 
print(f"S3 Bucket yang digunakan: {BUCKET}")

S3 Bucket yang digunakan: sagemaker-us-east-1-728358566341


In [6]:
def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]

role_arn = get_lab_role_arn()
print(f"Role ARN yang digunakan: {role_arn}")

Role ARN yang digunakan: arn:aws:iam::728358566341:role/LabRole


In [15]:
import tarfile
import os

local_model_path = "data/artifacts"
model_file = os.path.join(local_model_path, "best_model.joblib")
scaler_file = os.path.join(local_model_path, "scaler.joblib")

# Pastikan kedua file benar-benar ada
if os.path.exists(model_file) and os.path.exists(scaler_file):
    with tarfile.open("model.tar.gz", "w:gz") as tar:
        tar.add(model_file, arcname="best_model.joblib")
        tar.add(scaler_file, arcname="scaler.joblib")
        
    print("Done! model.tar.gz is ready to deploy.")
else:
    print(f"Error: File di direktori '{local_model_path}' tidak lengkap! Pastikan pipeline.py sudah selesai dijalankan.")

Done! model.tar.gz is ready to deploy.


In [16]:
s3_client = boto3.client("s3")
MODEL_S3_KEY = "credit-score-pipeline/model.tar.gz"

s3_client.upload_file("model.tar.gz", BUCKET, MODEL_S3_KEY)
print(f"Model berhasil diupload ke -> s3://{BUCKET}/{MODEL_S3_KEY}")

Model berhasil diupload ke -> s3://sagemaker-us-east-1-728358566341/credit-score-pipeline/model.tar.gz


In [17]:
import tarfile

with tarfile.open("model.tar.gz") as tar:
    for m in tar.getmembers():
        print(m.name)

best_model.joblib
scaler.joblib


In [19]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

# ---- EDIT THESE ---------------------------------------------------------
BUCKET = "sagemaker-us-east-1-728358566341"
MODEL_S3_KEY = "credit-score-pipeline/model.tar.gz"
ENDPOINT_NAME = "credit-score-v6"
# -------------------------------------------------------------------------

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"

def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]

def main() -> None:
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        source_dir = "inference",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint (5-8 minutes)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )

    print(
       f"\nEndpoint '{ENDPOINT_NAME}' is live in {REGION}.\n"
       f"Delete it before lab teardown: predictor.delete_endpoint()"
    )


if __name__ == "__main__":
    main()


Role:      arn:aws:iam::728358566341:role/LabRole
Model URI: s3://sagemaker-us-east-1-728358566341/credit-score-pipeline/model.tar.gz
Endpoint:  credit-score-v6

Deploying endpoint (5-8 minutes)...
-----!
Endpoint 'credit-score-v6' is live in us-east-1.
Delete it before lab teardown: predictor.delete_endpoint()


In [20]:
sm_client = boto3.client("sagemaker")
response = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
print(f"Status Endpoint Saat ini: {response['EndpointStatus']}")

Status Endpoint Saat ini: InService


In [32]:
print(f"Menghapus endpoint {ENDPOINT_NAME} untuk menghemat biaya...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Cleanup berhasil dilakukan!")
except Exception as e:
    print(f"Gagal/Sudah terhapus: {e}")

Menghapus endpoint credit-score-v1 untuk menghemat biaya...
Gagal/Sudah terhapus: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Cannot update in-progress endpoint "arn:aws:sagemaker:us-east-1:728358566341:endpoint/credit-score-v1".
